# 2. Classification: 1-NN with different similarities

In [ ]:
# --- Google Colab setup (does nothing when run locally) ---
import os, sys, subprocess

REPO_URL = "https://github.com/USERNAME/microlensing-dtw.git"  # <- edit to your repo
USE_DRIVE = True  # keep the repo (and results/, figures/) in Google Drive so they survive between notebooks

if "google.colab" in sys.modules:
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        REPO_DIR = "/content/drive/MyDrive/microlensing-dtw"
    else:
        REPO_DIR = "/content/microlensing-dtw"
    # Clone only if the project isn't there yet (you can also upload the folder to Drive by hand)
    if not os.path.isdir(os.path.join(REPO_DIR, "mldtw")):
        subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
    # numpy, matplotlib, scikit-learn, joblib are preinstalled on Colab
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tslearn", "VBMicrolensing"], check=True)
    os.chdir(os.path.join(REPO_DIR, "notebooks"))

os.makedirs("../results", exist_ok=True)
os.makedirs("../figures", exist_ok=True)
print("Working directory:", os.getcwd())

In [ ]:
import sys
sys.path.insert(0, "..")
import json
import numpy as np
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
from sklearn.metrics import accuracy_score
from tslearn.neighbors import KNeighborsTimeSeriesClassifier
from mldtw.simulate import make_dataset, centre

In [ ]:
X_train, y_train, _ = make_dataset(40, seed=3)
X_test, y_test, _ = make_dataset(30, seed=4)
Xc_train, Xc_test = centre(X_train), centre(X_test)

In [ ]:
band = {"global_constraint": "sakoe_chiba", "sakoe_chiba_radius": 24}
metrics = {
    "Euclid": (False, "euclidean", {}),
    "Euclid, centred": (True, "euclidean", {}),
    "DTW": (False, "dtw", {}),
    "DTW band": (False, "dtw", band),
    "Centred + DTW": (True, "dtw", {}),
    "Centred + DTW band": (True, "dtw", band),
}
for g in [0.01, 0.1, 1, 10]:
    metrics[f"Centred soft-DTW, gamma={g}"] = (True, "softdtw", {"gamma": g})

In [ ]:
def accuracy(centred, metric, params):
    A, B = (Xc_train, Xc_test) if centred else (X_train, X_test)
    knn = KNeighborsTimeSeriesClassifier(n_neighbors=1, metric=metric, metric_params=params)
    knn.fit(A, y_train)
    return accuracy_score(y_test, knn.predict(B))

scores = Parallel(n_jobs=len(metrics))(delayed(accuracy)(*m) for m in metrics.values())
acc = dict(zip(metrics, scores))
json.dump(acc, open("../results/accuracy.json", "w"), indent=1)
acc

In [ ]:
plt.barh(list(acc), list(acc.values()))
plt.xlim(0.5, 1)
plt.xlabel("1-NN accuracy")
plt.tight_layout()
plt.savefig("../figures/accuracy.png", dpi=150)